In [0]:
CREATE VIEW retail_oc.retail_semantic.retail_metric_view
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: retail_oc.retail_gold.fact_sales
comment: Retail sales metrics combining transaction data with customer, product, and calendar dimensions
joins:
  - name: customer
    source: retail_oc.retail_gold.dim_customer
    on: source.account_id = customer.customer_id
  - name: product
    source: retail_oc.retail_gold.dim_product
    on: source.product_id = product.product_id
  - name: calendar
    source: retail_oc.retail_gold.dim_calendar
    on: source.transaction_date = calendar.date
dimensions:
  - name: Transaction Date
    expr: calendar.date
    display_name: Transaction Date
    comment: Date when the transaction occurred
    format:
      type: date
      date_format: year_month_day
    synonyms:
      - date
      - sale date
      - order date
  - name: Year
    expr: calendar.year
    display_name: Year
    comment: Year of the transaction
  - name: Month
    expr: calendar.month_name
    display_name: Month
    comment: Month name of the transaction
  - name: Product Category
    expr: product.category
    display_name: Product Category
    comment: High-level product category
    synonyms:
      - category
  - name: Sales Channel
    expr: source.sales_channel
    display_name: Sales Channel
    comment: Channel through which the sale was made (ONLINE or STORE)
    synonyms:
      - channel
measures:
  - name: Total Revenue
    expr: SUM(source.net_amount)
    display_name: Total Revenue
    comment: Sum of net sales amount after discounts
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - revenue
      - sales
      - net sales
  - name: Transaction Count
    expr: COUNT(source.transaction_id)
    display_name: Transaction Count
    comment: Total number of transactions
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - order count
      - sales count
  - name: Average Order Value
    expr: MEASURE(`Total Revenue`) / MEASURE(`Transaction Count`)
    display_name: Average Order Value
    comment: Average revenue per transaction
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - AOV
      - avg order size
  - name: Total Quantity
    expr: SUM(source.quantity)
    display_name: Total Quantity
    comment: Total quantity of items sold
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
    synonyms:
      - units sold
      - quantity sold
  - name: Average Discount
    expr: AVG(source.discount_percentage)
    display_name: Average Discount
    comment: Average discount percentage applied
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 2
    synonyms:
      - discount rate
$$